# Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)

1) load data and inspect structure

In [ ]:
# The CSV has no header row, so this cell assigns inferred column names.
column_names = [
    'row_id', 'rank', 'spotify_track_id', 'artists_spotify', 'album_name',
    'track_name_spotify', 'popularity', 'duration_ms', 'explicit', 'danceability',
    'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
    'genre_spotify', 'lastfm_url', 'track_name_lastfm', 'artist_lastfm',
    'tags', 'tag_count', 'arousal', 'valence_emotion', 'dominance',
    'lastfm_track_id', 'spotify_track_id_dup', 'playlist_genre'
]

df = pd.read_csv('data.csv', header=None, names=column_names)

print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

2) data quality checks

In [ ]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame('missing_count')
      .assign(missing_pct=lambda x: (x['missing_count'] / len(df) * 100).round(2))
      .sort_values('missing_count', ascending=False)
)

missing_summary.head(15)

In [ ]:
exact_duplicates = df.duplicated().sum()
dup_spotify_id = df.duplicated(subset=['spotify_track_id']).sum()
dup_lastfm_track_id = df.duplicated(subset=['lastfm_track_id']).sum()

print('Exact duplicate rows:', exact_duplicates)
print('Duplicate spotify_track_id:', dup_spotify_id)
print('Duplicate lastfm_track_id:', dup_lastfm_track_id)

3) basic descriptive statistics

In [ ]:
numeric_cols = [
    'popularity', 'duration_ms', 'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'tag_count', 'arousal', 'valence_emotion', 'dominance'
]

df[numeric_cols].describe().T

4) simple univariate visualizations

In [ ]:
plot_cols = ['popularity', 'danceability', 'energy', 'acousticness', 'valence', 'tempo']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    axes[i].hist(df[col].dropna(), bins=30)
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['genre_spotify', 'playlist_genre', 'mode', 'explicit']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    top_vals = df[col].value_counts(dropna=False).head(10)
    top_vals.plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'Top values: {col}')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

5) relationship exploration

In [ ]:
corr_cols = [
    'popularity', 'danceability', 'energy', 'acousticness',
    'valence', 'tempo', 'arousal', 'valence_emotion', 'dominance'
]

corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(df['acousticness'], df['energy'], alpha=0.4)
axes[0].set_xlabel('acousticness')
axes[0].set_ylabel('energy')
axes[0].set_title('Acousticness vs Energy')

axes[1].scatter(df['danceability'], df['valence'], alpha=0.4)
axes[1].set_xlabel('danceability')
axes[1].set_ylabel('valence')
axes[1].set_title('Danceability vs Valence')

plt.tight_layout()
plt.show()

6) outlier snapshot with boxplots

In [ ]:
box_cols = ['popularity', 'duration_ms', 'tempo', 'loudness']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(box_cols):
    axes[i].boxplot(df[col].dropna(), vert=True)
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

7) frequency view of most common artists

In [ ]:
top_artists = df['artist_lastfm'].value_counts().head(15)

plt.figure(figsize=(10, 5))
top_artists.plot(kind='bar')
plt.title('Top 15 Artists by Track Count')
plt.xlabel('Artist')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

8) EDA findings in paragraph form